# Week 5: Dynamic Hedging Simulation

This notebook extends the single-path delta hedging table from [Week 4](../week4_greeks/greeks.ipynb) 
to 1000 independent GBM paths. Where Week 4 asked *"does the hedge work?"*, 
Week 5 asks *"what does the distribution of hedging outcomes look like across many paths?"*

## Setup

Same parameters as Week 4's hedging table — same stock, same option, same 20-week horizon.

In [ ]:
from scipy.stats import norm
from math import log, sqrt, exp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Setup inputs
s_0 = 49
k = 50
r = 0.05
mu = 0.15
sigma = 0.30
T = 20/52        # 20 weeks in years
N = 20     # one step per week
dt = T / N # = 1/52
num_paths = 1000
n_options = 100000

# Generating random draws
Z = np.random.standard_normal((num_paths, N))

## Generating 1000 GBM Paths

Recall from Week 1 that GBM evolves the stock price as:

$$S_{t+1} = S_t \cdot \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)\Delta t + \sigma \sqrt{\Delta t} \cdot Z\right)$$

Each simulation produces $N+1$ price points — weekly snapshots from $S_0$ through $S_{20}$. 
We generate 1000 independent paths, each a possible future for the stock.

In [ ]:
#  Building price paths in GBM
prices = np.zeros ((num_paths, N+1))
prices[:, 0] = s_0         
for t in range(1, N+1):
            prices[:, t] = prices[:, t-1] * np.exp((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * Z[:, t-1])

# Show a sample of values from GBM walk
np.set_printoptions(precision=2, suppress=True)
print(prices[:3])

[[49.   50.86 51.7  49.56 50.33 52.91 51.76 53.96 48.81 49.07 50.98 49.7
  51.2  50.22 48.94 45.05 44.51 44.11 45.56 46.19 44.68]
 [49.   51.28 53.08 51.85 56.12 57.15 58.82 56.61 59.35 58.38 57.75 58.04
  59.23 58.98 59.34 57.49 60.04 64.71 61.66 60.58 61.62]
 [49.   52.1  52.71 51.53 52.77 54.69 55.56 61.24 66.14 64.85 65.09 65.12
  66.39 65.51 70.28 70.36 72.17 70.23 68.56 74.05 70.53]]


In [ ]:

# Function for d1 and d2
def d_1(sigma, s_0, k, r, T):
    d_1 = (log(s_0/k)+(r+(sigma**2)/2)*T)/((sigma)*sqrt(T))
    return d_1

def d_2(sigma, s_0, k, r, T): 
    return d_1(sigma, s_0, k, r, T) - (sigma)*sqrt(T)

# Identifying option prices using BSM
def compute_delta_hedge(sigma, S, k, r, T_remaining):
    if T_remaining <= 0:
        return 1.0 if S > k else 0.0
    return norm.cdf(d_1(sigma, S, k, r, T_remaining))

# BSM call pricer only
def bsm(sigma, s_0, k, r, T):
    d1 = d_1(sigma, s_0, k, r, T)
    d2 = d_2(sigma, s_0, k, r, T)
    opt_price = s_0 * norm.cdf(d1) - k * exp(-r * T) * norm.cdf(d2)
    return opt_price

# Compute options cost with BSM:
bsm_price = bsm(sigma, s_0, k, r, T) * n_options

# Build pnl list
pnl_list = []

for i in range(num_paths):
    stock_prices = prices[i]
    
    hedge_account = bsm_price  # start with premium collected
    shares_held = 0.0
    
    for t in range(N):
        # Grow account by one period of interest
        hedge_account *= exp(r * dt)
        
        # Compute new delta
        T_remaining = T - t * dt
        new_delta = compute_delta_hedge(sigma, stock_prices[t], k, r, T_remaining)
        
        # Rebalance: buy/sell shares
        shares_to_trade = new_delta * n_options - shares_held
        hedge_account -= shares_to_trade * stock_prices[t]
        shares_held = new_delta * n_options  # already in share units
    
    # Expiry: liquidate shares, pay option payoff
    hedge_account += shares_held * stock_prices[-1]
    option_payoff = max(stock_prices[-1] - k, 0) * n_options
    hedge_account -= option_payoff
    
    pnl_list.append(hedge_account)



t=0: delta=0.5350, shares_to_trade=53,498.30, hedge_account=-2,260,096.33
t=1: delta=0.6125, shares_to_trade=7,756.50, hedge_account=-2,656,802.36
t=2: delta=0.6466, shares_to_trade=3,405.46, hedge_account=-2,835,433.66
